# Ingredient extraction and normalisation

Uses Ollama (Gemma3 4B) to extract structured ingredients from recipe text and optionally normalise names/units. Compares LLM output to Cleaned-Ingredients for sanity checks.

In [1]:
import sys
from pathlib import Path

cwd = Path.cwd()
ROOT = cwd if (cwd / "dataset").exists() else cwd.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src.load_recipes import (
    load_recipes,
    get_recipe_by_name,
    get_recipe_text_for_llm,
    parse_cleaned_ingredients,
    RECIPE_NAME_COL,
    CLEANED_INGREDIENTS_COL,
)
from src.recipe_to_ingredients import recipe_name_to_ingredients

## 1. Load recipes and pick test names

In [2]:
df_recipes = load_recipes()
print("Recipes:", len(df_recipes))

# Use first recipe and a couple more for sanity check
test_names = [
    df_recipes[RECIPE_NAME_COL].iloc[0],
    df_recipes[RECIPE_NAME_COL].iloc[100],
    df_recipes[RECIPE_NAME_COL].iloc[500],
]
print("Test recipes:", test_names)

Recipes: 5938
Test recipes: ['Masala Karela Recipe', 'Capsicum Masala Poriyal (Recipe in Hindi)', 'Foxtail Millet Pongal Recipe']


## 2. Extract + normalise ingredients (LLM)

In [3]:
results = []
for name in test_names:
    ingredients, row = recipe_name_to_ingredients(
        name,
        df_recipes=df_recipes,
        include_instructions=True,
        normalise=True,
        use_fallback_on_failure=True,
    )
    results.append({"name": name, "ingredients": ingredients, "row": row})
    print(f"\n{name}")
    print(f"  Extracted count: {len(ingredients)}")
    for i, ing in enumerate(ingredients[:8]):
        print(f"    {i+1}. {ing['ingredient']} - {ing['quantity']} {ing['unit']}")
    if len(ingredients) > 8:
        print(f"    ... and {len(ingredients) - 8} more")


Masala Karela Recipe
  Extracted count: 11
    1. red chilli powder - 1.0 tablespoon
    2. gram flour - 3.0 tablespoon
    3. cumin seeds - 2.0 teaspoon
    4. coriander powder - 1.0 tablespoon
    5. turmeric powder - 2.0 teaspoon
    6. salt - 1.0 to taste
    7. amchur - 1.0 tablespoon
    8. karela - 6.0 piece
    ... and 3 more

Capsicum Masala Poriyal (Recipe in Hindi)
  Extracted count: 14
    1. salt - 1.0 pinch
    2. peanuts - 1.0 piece
    3. coconut - 1.0 piece
    4. capsicum - 1.0 piece
    5. capsicum - 1.0 piece
    6. capsicum - 1.0 piece
    7. mustard seeds - 1.0 teaspoon
    8. curry leaves - 1.0 piece
    ... and 6 more

Foxtail Millet Pongal Recipe
  Extracted count: 12
    1. black peppercorns - 2.0 teaspoon
    2. green chilli - 2.0 piece
    3. salt - 1.0 to taste
    4. cashew nuts - 8.0 piece
    5. cumin seeds - 1.0 teaspoon
    6. ginger - 1.0 teaspoon
    7. ghee - 2.0 tablespoon
    8. foxtail millet - 0.5 cup
    ... and 4 more


## 3. Sanity check: compare to Cleaned-Ingredients

In [4]:
for r in results:
    row = r["row"]
    if row is None:
        continue
    cleaned = parse_cleaned_ingredients(row.get(CLEANED_INGREDIENTS_COL))
    extracted_names = [x["ingredient"] for x in r["ingredients"]]
    print(f"Recipe: {r['name'][:50]}...")
    print(f"  Cleaned-Ingredients count: {len(cleaned)}")
    print(f"  LLM extracted count:      {len(extracted_names)}")
    # Overlap: normalise for comparison (lowercase, strip)
    cleaned_set = {c.strip().lower() for c in cleaned}
    extracted_set = {e.strip().lower() for e in extracted_names}
    overlap = cleaned_set & extracted_set
    print(f"  Overlap (by name):        {len(overlap)} / {len(cleaned_set)}")
    if cleaned_set - extracted_set:
        print(f"  In cleaned but not LLM:   {list(cleaned_set - extracted_set)[:5]}")
    if extracted_set - cleaned_set:
        print(f"  In LLM but not cleaned:   {list(extracted_set - cleaned_set)[:5]}")
    print()

Recipe: Masala Karela Recipe...
  Cleaned-Ingredients count: 10
  LLM extracted count:      11
  Overlap (by name):        5 / 10
  In cleaned but not LLM:   ['amchur (dry mango powder)', 'sunflower oil', 'karela (bitter gourd pavakkai)', 'gram flour (besan)', 'cumin seeds (jeera)']
  In LLM but not cleaned:   ['vegetable oil', 'karela', 'gram flour', 'amchur', 'water']

Recipe: Capsicum Masala Poriyal (Recipe in Hindi)...
  Cleaned-Ingredients count: 14
  LLM extracted count:      14
  Overlap (by name):        6 / 14
  In cleaned but not LLM:   ['oil', 'mustard', 'capsicum (green)', 'peanuts roast', 'coriander (dhania) leaves']
  In LLM but not cleaned:   ['vegetable oil', 'capsicum', 'peanuts', 'mustard seeds', 'garlic']

Recipe: Foxtail Millet Pongal Recipe...
  Cleaned-Ingredients count: 12
  LLM extracted count:      12
  Overlap (by name):        8 / 12
  In cleaned but not LLM:   ['cumin seeds (jeera)', 'yellow moong dal (split)', 'green chillies', 'asafoetida (hing)']
  In LLM

## 4. Run on 5 recipes (sanity batch)

In [5]:
sample = df_recipes.sample(n=5, random_state=42)
names_5 = sample[RECIPE_NAME_COL].tolist()
success = 0
for name in names_5:
    ingredients, _ = recipe_name_to_ingredients(name, df_recipes=df_recipes, normalise=True)
    if ingredients:
        success += 1
print(f"Extraction success: {success} / {len(names_5)} recipes")

Extraction success: 5 / 5 recipes
